# DrawNet
CS4085 Deep Learning

**Team members**
- Yara Mohammad
- Lama Alturki
- Amal Shuaileh

This notebook trains M0 (small CNN), M1 (same CNN with augmentation), and converts M1 to M2 (full INT8). Run cells in order in Google Colab. Results are calculated by the code, not filled in beforehand.

**Start:** upload this notebook to Colab, choose a GPU if available, then run the cells. Full mode uses 10 classes, 3,000 images per class and 15 epochs. Set `QUICK_RUN=True` for a pipeline check only. Quick-run results are not final research evidence.

The demo is optional and uses a temporary Gradio link only when `SHARE_DEMO=True`. No account keys are needed. For persistent files, enable `USE_DRIVE`. Colab collaborators have separate runtimes.


In [ ]:
# Install the versions used for this notebook. Restart the runtime if requested.
%pip -q install tensorflow==2.16.1 numpy==1.26.4 scikit-learn==1.5.2 pandas==2.2.3 matplotlib==3.9.2 pillow==11.3.0 psutil==6.1.1 gradio==5.49.1


## 1 Setup


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import json, time, random, platform, hashlib, shutil, gc
from pathlib import Path
from urllib.request import urlretrieve
from urllib.parse import quote
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import psutil
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from IPython.display import display

SEED = 42
QUICK_RUN = False
USE_DRIVE = False
RUN_DEMO = True
SHARE_DEMO = False
class_names = ['tree', 'cup', 'bicycle', 'apple', 'cat', 'car', 'house', 'fish', 'flower', 'umbrella']
SAMPLES_PER_CLASS = 150 if QUICK_RUN else 3000
EPOCHS = 1 if QUICK_RUN else 15
BATCH_SIZE = 64
LEARNING_RATE = 0.001
ROTATION = 0.05  # fraction of one full turn, so at most 18 degrees
TRANSLATION = 0.08

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/DrawNet_results')
else:
    OUT = Path('DrawNet_results')
OUT.mkdir(parents=True, exist_ok=True)
DATA = OUT / 'data'
DATA.mkdir(exist_ok=True)

def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)
set_seed()
tf.config.experimental.enable_op_determinism()
try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass
config = dict(seed=SEED, quick_run=QUICK_RUN, classes=class_names,
              samples_per_class=SAMPLES_PER_CLASS, epochs=EPOCHS,
              batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
              rotation=ROTATION, translation=TRANSLATION)
(OUT / 'config.json').write_text(json.dumps(config, indent=2))
environment = dict(python=platform.python_version(), tensorflow=tf.__version__,
                   platform=platform.platform(), cpu=platform.processor(),
                   logical_cpus=os.cpu_count(), gpu=[d.name for d in tf.config.list_physical_devices('GPU')])
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
print(environment)
print('Mode:', 'PIPELINE CHECK' if QUICK_RUN else 'FULL EXPERIMENT')


## 2 Yara | Load and inspect the data
Files come from the official Quick, Draw! bitmap collection. Full class files are cached locally, so the first download can take time. We sample reproducibly, reject blank images and remove exact duplicate images across classes **before splitting**. This avoids identical bitmaps appearing in different splits. Conflicting identical drawings are excluded by keeping only the first occurrence; the removal counts are recorded. This does not rule out near-duplicates or artist overlap.


In [ ]:
images, labels = [], []
seen = set()
audit = []
for label, name in enumerate(class_names):
    path = DATA / (name + '.npy')
    if not path.exists():
        url = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/' + quote(name) + '.npy'
        print('Downloading', name)
        temporary = path.with_suffix('.part')
        for attempt in range(3):
            try:
                urlretrieve(url, temporary)
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2)
        temporary.replace(path)
    drawings = np.load(path, mmap_mode='r', allow_pickle=False)
    rng = np.random.default_rng(SEED + label)
    order = rng.permutation(len(drawings))
    selected = []
    rejected = 0
    for index in order:
        drawing = np.asarray(drawings[index], dtype=np.uint8).reshape(28, 28)
        key = hashlib.sha256(drawing.tobytes()).digest()
        if drawing.max() == 0 or key in seen:
            rejected += 1
            continue
        seen.add(key)
        selected.append(drawing.copy())
        if len(selected) == SAMPLES_PER_CLASS:
            break
    if len(selected) != SAMPLES_PER_CLASS:
        raise ValueError('Not enough unique nonblank drawings for ' + name)
    images.extend(selected)
    labels.extend([label] * len(selected))
    audit.append(dict(category=name, selected=len(selected), rejected=rejected))
X_uint8 = np.stack(images)
y = np.array(labels, dtype=np.int64)
del images, labels, seen
pd.DataFrame(audit).to_csv(OUT / 'data_audit.csv', index=False)
display(pd.DataFrame(audit))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_uint8[np.flatnonzero(y == i)[0]], cmap='gray', vmin=0, vmax=255)
    ax.set_title(class_names[i])
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT / 'sample_drawings.png', dpi=180)
plt.show()


## 3 Yara | Split and normalize


In [ ]:
all_indices = np.arange(len(y))
train_ids, remaining = train_test_split(all_indices, test_size=0.30, stratify=y, random_state=SEED)
val_ids, test_ids = train_test_split(remaining, test_size=0.50, stratify=y[remaining], random_state=SEED)
X = X_uint8.astype('float32')[..., np.newaxis] / 255.0
X_train, y_train = X[train_ids], y[train_ids]
X_val, y_val = X[val_ids], y[val_ids]
X_test, y_test = X[test_ids], y[test_ids]
assert not set(train_ids) & set(test_ids)
assert not set(val_ids) & set(test_ids)
np.savez_compressed(OUT / 'splits.npz', X_train=X_train, y_train=y_train,
                    X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test,
                    train_ids=train_ids, val_ids=val_ids, test_ids=test_ids)
(OUT / 'classes.json').write_text(json.dumps(class_names))
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)


## 4 Yara | Baseline M0
Two convolutional blocks, global average pooling and a softmax classifier. The same initial weights and shuffled training order are used for M0 and M1. Both train for the same number of epochs; their best validation-loss checkpoints are loaded. No test evaluation occurs until all settings and checkpoints are fixed.


In [ ]:
def build_cnn():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(len(class_names), activation='softmax')
    ])

set_seed()
initial_model = build_cnn()
initial_weights = initial_model.get_weights()
model_m0 = build_cnn()
model_m0.set_weights(initial_weights)
model_m0.summary()

def make_train_data(augmentation=None):
    ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    ds = ds.shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True).batch(BATCH_SIZE)
    if augmentation is not None:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=1)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 1
    return ds.with_options(options).prefetch(1)

training_info = {}
def train_model(model, name, augmentation=None):
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    checkpoint = OUT / (name + '.keras')
    callback = tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor='val_loss', save_best_only=True)
    process = psutil.Process()
    rss_start = process.memory_info().rss / 1024**2
    start = time.perf_counter()
    history = model.fit(make_train_data(augmentation), epochs=EPOCHS,
                        validation_data=(X_val, y_val), callbacks=[callback], verbose=2)
    training_info[name] = dict(training_seconds=time.perf_counter() - start,
                               process_rss_start_MiB=rss_start,
                               process_rss_end_MiB=process.memory_info().rss / 1024**2,
                               best_validation_epoch=int(np.argmin(history.history['val_loss'])) + 1)
    (OUT / (name + '_history.json')).write_text(json.dumps(history.history, indent=2))
    return tf.keras.models.load_model(checkpoint), history.history

model_m0, history_m0 = train_model(model_m0, 'm0')


## 5 Lama | Augmentation and M1


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(ROTATION, fill_mode='constant', fill_value=0, seed=SEED),
    tf.keras.layers.RandomTranslation(TRANSLATION, TRANSLATION, fill_mode='constant', fill_value=0, seed=SEED + 1)
])
preview = augmentation(X_train[:8], training=True).numpy()
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(X_train[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[1, i].imshow(preview[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].axis('off')
axes[0, 0].set_title('Original')
axes[1, 0].set_title('Augmented')
plt.tight_layout()
plt.savefig(OUT / 'augmentation.png', dpi=180)
plt.show()
# Recreate after preview so preview does not change the training random state.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(ROTATION, fill_mode='constant', fill_value=0, seed=SEED),
    tf.keras.layers.RandomTranslation(TRANSLATION, TRANSLATION, fill_mode='constant', fill_value=0, seed=SEED + 1)
])
set_seed()
model_m1 = build_cnn()
model_m1.set_weights(initial_weights)
assert model_m0.count_params() == model_m1.count_params()
model_m1, history_m1 = train_model(model_m1, 'm1', augmentation)
(OUT / 'training_info.json').write_text(json.dumps(training_info, indent=2))


## 6 Training visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in [('M0', history_m0), ('M1', history_m1)]:
    epochs = range(1, len(history['loss']) + 1)
    axes[0].plot(epochs, history['accuracy'], label=name + ' train')
    axes[0].plot(epochs, history['val_accuracy'], '--', label=name + ' validation')
    axes[1].plot(epochs, history['loss'], label=name + ' train')
    axes[1].plot(epochs, history['val_loss'], '--', label=name + ' validation')
for ax, title in zip(axes, ['Accuracy', 'Loss']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'learning_curves.png', dpi=180)
plt.show()


## 7 Amal | Float exports and full INT8 M2
Augmentation is outside the inference model, so no random operations enter the exported model. Calibration uses training data only. Conversion allows integer built-in operators and requires INT8 input and output. Bias and shape tensors can still use integer types such as INT32; “full INT8” does not mean every metadata tensor is INT8.


In [ ]:
def export_tflite(model, name, integer=False):
    # SavedModel route avoids Keras conversion differences.
    folder = OUT / (name + '_savedmodel')
    model.export(str(folder))
    converter = tf.lite.TFLiteConverter.from_saved_model(str(folder))
    if integer:
        rng = np.random.default_rng(SEED)
        ids = rng.choice(len(X_train), min(500, len(X_train)), replace=False)
        def representative_data():
            for i in ids:
                yield [X_train[i:i+1].astype(np.float32)]
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
    result = converter.convert()
    path = OUT / (name + '.tflite')
    path.write_bytes(result)
    return path

paths = {'M0': export_tflite(model_m0, 'm0_float'),
         'M1': export_tflite(model_m1, 'm1_float'),
         'M2': export_tflite(model_m1, 'm2_int8', integer=True)}

def get_interpreter(path):
    interpreter = tf.lite.Interpreter(model_path=str(path), num_threads=1)
    interpreter.allocate_tensors()
    return interpreter

check = get_interpreter(paths['M2'])
assert check.get_input_details()[0]['dtype'] == np.int8
assert check.get_output_details()[0]['dtype'] == np.int8
assert all(t['dtype'] not in (np.float32, np.float16, np.float64) for t in check.get_tensor_details())
print('M2 has integer tensors and INT8 input/output.')


## 8 Amal | Predictions and fair CPU benchmark


In [ ]:
def prepare_input(x, detail):
    if np.issubdtype(detail['dtype'], np.integer):
        scale, zero = detail['quantization']
        if scale <= 0:
            raise ValueError('Invalid input quantization scale')
        limits = np.iinfo(detail['dtype'])
        return np.clip(np.round(x / scale + zero), limits.min, limits.max).astype(detail['dtype'])
    return x.astype(detail['dtype'])

def read_output(values, detail):
    if np.issubdtype(detail['dtype'], np.integer):
        scale, zero = detail['quantization']
        return (values.astype(np.float32) - zero) * scale
    return values

def predict_lite(interpreter, data):
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    predictions = []
    for x in data:
        interpreter.set_tensor(inp['index'], prepare_input(x[np.newaxis], inp))
        interpreter.invoke()
        predictions.append(read_output(interpreter.get_tensor(out['index'])[0], out))
    return np.array(predictions)

def benchmark(path):
    # Same runtime, one thread and batch size 1. Timing excludes Python preprocessing.
    process = psutil.Process()
    gc.collect()
    rss_before = process.memory_info().rss
    interpreter = get_interpreter(path)
    inp = interpreter.get_input_details()[0]
    samples = [prepare_input(x[np.newaxis], inp) for x in X_val[:min(100, len(X_val))]]
    for i in range(30):
        interpreter.set_tensor(inp['index'], samples[i % len(samples)])
        interpreter.invoke()
    times = []
    for i in range(300):
        interpreter.set_tensor(inp['index'], samples[i % len(samples)])
        start = time.perf_counter_ns()
        interpreter.invoke()
        times.append((time.perf_counter_ns() - start) / 1e6)
    rss_after = process.memory_info().rss
    return dict(latency_median_ms=float(np.median(times)), latency_p90_ms=float(np.percentile(times, 90)),
                process_rss_MiB=rss_after / 1024**2,
                observed_rss_change_MiB=(rss_after - rss_before) / 1024**2)

# Analytical MAC count for this exact architecture, batch size 1.
# Conv1: 28*28*16*(3*3*1), Conv2: 14*14*32*(3*3*16), Dense: 32*classes.
MACS = 28*28*16*9 + 14*14*32*3*3*16 + 32*len(class_names)
# Convention: FLOPs = 2 * MACs, excludes pooling, bias, activations and GAP.
rows, probabilities = [], {}
for name, path in paths.items():
    interpreter = get_interpreter(path)
    probabilities[name] = predict_lite(interpreter, X_test)
    predictions = probabilities[name].argmax(axis=1)
    np.save(OUT / (name + '_test_predictions.npy'), predictions)
    row = dict(model=name, accuracy=accuracy_score(y_test, predictions),
               macro_f1=f1_score(y_test, predictions, average='macro'),
               parameters=model_m0.count_params(), size_KiB=path.stat().st_size/1024,
               MACs=MACS, FLOPs_2xMACs=2*MACS)
    row.update(benchmark(path))
    rows.append(row)
results = pd.DataFrame(rows).set_index('model')
results.to_csv(OUT / 'results.csv')
display(results)
print('Memory is whole-process RSS in this notebook, NOT an isolated model peak or tensor arena measurement.')
print('RSS changes can be affected by allocator reuse. MAC/FLOP counts are analytical, not measured CPU instructions.')
print('INT8 changes representation, not the architecture parameter count or theoretical MAC count.')
# Confirm float export predictions are close to Keras on validation data.
for name, model in [('M0', model_m0), ('M1', model_m1)]:
    lite = predict_lite(get_interpreter(paths[name]), X_val[:32])
    keras = model(X_val[:32], training=False).numpy()
    np.testing.assert_allclose(lite, keras, atol=1e-4, rtol=1e-3)
print('Float export parity checks passed.')


## 9 All members | Charts and error analysis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['#7b71c7', '#38a6a0', '#d38c4c']
for ax, metric, title in zip(axes, ['accuracy', 'size_KiB', 'latency_median_ms'],
                            ['Test accuracy', 'File size (KiB)', 'CPU invoke latency (ms)']):
    ax.bar(results.index, results[metric], color=colors)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(OUT / 'model_comparison.png', dpi=180)
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, name in zip(axes, paths):
    prediction = probabilities[name].argmax(axis=1)
    matrix = confusion_matrix(y_test, prediction, labels=range(len(class_names)))
    ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(ax=ax, colorbar=False, xticks_rotation=90)
    ax.set_title(name)
    report = classification_report(y_test, prediction, target_names=class_names, output_dict=True, zero_division=0)
    (OUT / (name + '_classification_report.json')).write_text(json.dumps(report, indent=2))
plt.tight_layout()
plt.savefig(OUT / 'confusion_matrices.png', dpi=180)
plt.show()
wrong = np.flatnonzero(probabilities['M1'].argmax(axis=1) != y_test)[:12]
if len(wrong):
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    for ax in axes.flat:
        ax.axis('off')
    for ax, i in zip(axes.flat, wrong):
        ax.imshow(X_test[i, :, :, 0], cmap='gray')
        ax.set_title('True: ' + class_names[y_test[i]] + '\nPred: ' + class_names[probabilities['M1'][i].argmax()])
    plt.tight_layout()
    plt.savefig(OUT / 'm1_errors.png', dpi=180)
    plt.show()


## 10 Hypotheses | Report measured outcomes honestly


In [ ]:
a0, a1, a2 = results['accuracy']
gain = a1 - a0
size_reduction = 100 * (1 - results.loc['M2', 'size_KiB'] / results.loc['M1', 'size_KiB'])
h1 = bool(gain >= 0.01)
h2 = bool(size_reduction >= 60 and gain > 0 and (a2 - a0) >= 0.5 * gain)
summary = dict(H1_supported_this_run=h1, H2_supported_this_run=h2,
               M1_gain_percentage_points=float(100*gain),
               M2_accuracy_change_vs_M1_percentage_points=float(100*(a2-a1)),
               size_reduction_percent=float(size_reduction), quick_run=QUICK_RUN)
(OUT / 'hypotheses.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('Exploratory targets: H1 +1 percentage point; H2 >=60% smaller and retains >=half of positive M1 gain.')
print('One training seed is not proof of statistical significance. Repeat with predeclared seeds for stronger evidence.')


## 11 Interactive drawing demo
Draw one of the listed classes on the white canvas. Predictions are forced choices among these classes, not an unknown-object detector. Confidence is not a guarantee of correctness. Hand drawings may differ from training data. The app compares all three exported models. Do not use demo feedback to tune models after final test evaluation.


In [ ]:
def preprocess_drawing(editor_value):
    if editor_value is None:
        raise ValueError('Draw an object first.')
    picture = editor_value.get('composite') if isinstance(editor_value, dict) else editor_value
    if picture is None:
        raise ValueError('Draw an object first.')
    image = Image.fromarray(np.asarray(picture).astype('uint8')) if not isinstance(picture, Image.Image) else picture
    rgba = image.convert('RGBA')
    white = Image.new('RGBA', rgba.size, 'white')
    white.alpha_composite(rgba)
    # Quick Draw bitmaps have light ink on a dark background.
    ink = ImageOps.invert(white.convert('RGB').convert('L'))
    array = np.asarray(ink)
    points = np.argwhere(array > 30)
    if len(points) < 5:
        raise ValueError('The canvas is empty.')
    top, left = points.min(axis=0)
    bottom, right = points.max(axis=0) + 1
    ink = ink.crop((int(left), int(top), int(right), int(bottom)))
    scale = 24 / max(ink.size)
    size = (max(1, round(ink.width*scale)), max(1, round(ink.height*scale)))
    ink = ink.resize(size, Image.Resampling.LANCZOS)
    canvas = Image.new('L', (28, 28), 0)
    canvas.paste(ink, ((28-size[0])//2, (28-size[1])//2))
    return np.asarray(canvas, dtype=np.float32)[..., None] / 255.0

demo_interpreters = {name: get_interpreter(path) for name, path in paths.items()}
def recognize_drawing(value):
    import gradio as gr
    try:
        image = preprocess_drawing(value)
    except ValueError as error:
        raise gr.Error(str(error))
    predictions = []
    for name in ['M0', 'M1', 'M2']:
        scores = predict_lite(demo_interpreters[name], image[np.newaxis])[0]
        predictions.append({label: float(score) for label, score in zip(class_names, scores)})
    return *predictions, image[:, :, 0]

if RUN_DEMO:
    import gradio as gr
    with gr.Blocks(title='DrawNet') as demo:
        gr.Markdown('# DrawNet\nDraw an object and compare three small neural network models.')
        gr.Markdown('Classes: ' + ', '.join(class_names))
        with gr.Row():
            sketch = gr.Sketchpad(type='numpy', image_mode='RGBA', canvas_size=(360, 360),
                                 brush=gr.Brush(colors=['#000000'], color_mode='fixed', default_size=8),
                                 label='Draw here')
            with gr.Column():
                outputs = [gr.Label(num_top_classes=3, label=name) for name in ['M0 Baseline', 'M1 Augmentation', 'M2 INT8']]
                processed = gr.Image(label='28 by 28 model input', image_mode='L')
        button = gr.Button('Recognize', variant='primary')
        button.click(recognize_drawing, inputs=sketch, outputs=outputs+[processed], concurrency_limit=1)
        gr.Dataframe(value=results.reset_index(), interactive=False, label='Measured results')
    demo.launch(share=SHARE_DEMO)


## 12 Save and download


In [ ]:
# The archive contains models, figures, configuration, splits and real results.
# Raw downloaded class files are excluded to keep it manageable.
archive_dir = OUT / 'export'
archive_dir.mkdir(exist_ok=True)
for path in OUT.iterdir():
    if path.is_file() and path.suffix in ['.keras', '.tflite', '.json', '.csv', '.png', '.npz', '.npy']:
        shutil.copy2(path, archive_dir / path.name)
archive = shutil.make_archive(str(OUT / 'DrawNet_results'), 'zip', archive_dir)
print('Saved:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Open the results folder to get your files.')


## Discussion and sources
Write your own interpretation after running: Which classes are confused? Did augmentation help? Did INT8 preserve the gain? Was it faster on the measured CPU? Explain negative results too. Report exact configuration, hardware and limitations. Do not report quick-run outputs as a full experiment.

**Data:** [Google Quick, Draw! dataset](https://github.com/googlecreativelab/quickdraw-dataset). Drawings are distributed under CC BY 4.0; follow attribution requirements. This notebook uses the bitmap format.

**Implementation:** [TensorFlow augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation), [Full integer quantization](https://www.tensorflow.org/lite/performance/post_training_integer_quant), [Gradio ImageEditor](https://www.gradio.app/docs/gradio/imageeditor).

**Research:** [Yu et al. 2015](https://doi.org/10.5244/C.29.7), [Shorten and Khoshgoftaar 2019](https://doi.org/10.1186/s40537-019-0197-0), [Jacob et al. 2018](https://openaccess.thecvf.com/content_cvpr_2018/html/Jacob_Quantization_and_Training_CVPR_2018_paper.html). Complete the assignment's five-paper literature requirement in the report.

**AI use:** This code was drafted with ChatGPT assistance. Team members should run it, understand the implementation, verify sources, and disclose assistance in the submission.
